# Lab 11: Cascadia Slow Slip Events — From Time Series to Source Parameters

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/11_slow_slip_lab.ipynb)

## Introduction

In Lab 2 you used station ALBH to identify the 2015–2016 Cascadia slow slip event in a GNSS residual time series. In this lab you extend that analysis across a network of coastal stations, inverting for the spatial distribution of slip on the Cascadia subduction interface during the **August–September 2024** ETS event.

This is the full source-characterisation pipeline: extract offsets, build **G**, regularise, choose λ with the L-curve, assess resolution, and compute moment, magnitude, and stress drop.

## Learning objectives

By the end, you will be able to:

- extract SSE offsets from GNSS time series at multiple stations and propagate uncertainties
- load and filter a triangulated Cascadia fault mesh to the ETS depth range
- build a Green's function matrix using `cutde`
- run a smoothed, non-negative least-squares inversion and choose λ with the L-curve
- compute and interpret the resolution matrix
- compute seismic moment, moment magnitude, and stress drop from a slip model

## Notebook outline

- [Setup](#setup)
- [Part I: Extract SSE offsets](#part-i-extract-sse-offsets)
- [Part II: Cascadia fault mesh](#part-ii-cascadia-fault-mesh)
- [Part III: Green's function matrix](#part-iii-greens-function-matrix)
- [Part IV: Invert for slip](#part-iv-invert-for-slip)
- [Part V: Resolution matrix](#part-v-resolution-matrix)
- [Part VI: Moment, magnitude, and stress drop](#part-vi-moment-magnitude-and-stress-drop)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup

Builds on code from Labs 2, 9, and 10. Run this cell before continuing.

In [ ]:
%pip install -q cutde cartopy

import os, io, urllib.request
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import requests
import cutde.halfspace as hs
from scipy.optimize import nnls
from scipy.linalg import lstsq
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.grid': True, 'grid.alpha': 0.2})

# ── Physical constants ────────────────────────────────────────────────────────
NU    = 0.25        # Poisson ratio
MU    = 30e9        # shear modulus (Pa) — lower crust / mantle wedge
SCALE = 111_195.0   # metres per degree latitude
REF_LON, REF_LAT = -124.5, 48.0   # local Cartesian reference point

# ── Coordinate conversion ─────────────────────────────────────────────────────
def lonlat_to_m(lon, lat):
    """Convert lon/lat (degrees) to local Cartesian metres."""
    x = (np.asarray(lon) - REF_LON) * SCALE * np.cos(np.radians(REF_LAT))
    y = (np.asarray(lat) - REF_LAT) * SCALE
    return x, y

# ── Time-series helpers (Lab 2) ───────────────────────────────────────────────
def build_design_matrix(t, seasonal=True, t0=None):
    """Offset + velocity + optional annual/semiannual columns.
    Pass t0 explicitly to keep the same reference epoch when fitting
    on a subset and evaluating on the full record."""
    if t0 is None:
        t0 = np.nanmean(t)
    dt = t - t0
    cols = [np.ones_like(t), dt]
    if seasonal:
        for period in [1.0, 0.5]:
            omega = 2 * np.pi / period
            cols += [np.cos(omega * t), np.sin(omega * t)]
    return np.column_stack(cols)

def fit_residual(t, d, sigma, seasonal=True):
    """Weighted least-squares fit; return residual (mm).
    Uses element-wise weighting — no full N×N weight matrix constructed."""
    valid = np.isfinite(t) & np.isfinite(d) & np.isfinite(sigma) & (sigma > 0)
    G_v   = build_design_matrix(t[valid], seasonal=seasonal)
    w     = 1.0 / sigma[valid]          # weight per observation
    A     = G_v * w[:, None]
    b     = d[valid] * w
    params, _, _, _ = lstsq(A, b)
    G_all = build_design_matrix(t, seasonal=seasonal)
    return d - G_all @ params

# ── Regularised inversion helpers (Lab 10) ────────────────────────────────────
def build_spatial_laplacian(centroids_m, radius_m):
    """Graph Laplacian: neighbours are patches within radius_m of each other."""
    n     = len(centroids_m)
    dists = np.sqrt(((centroids_m[:, None] - centroids_m[None, :])**2).sum(axis=2))
    adj   = (dists > 0) & (dists <= radius_m)
    return np.diag(adj.sum(axis=1).astype(float)) - adj.astype(float)

def solve_regularized(G, d_obs, sig_d, lam, L=None, nonneg=True):
    """Weighted, (smoothed or damped) NNLS or LS.
    Returns (m, d_pred, normalized_rms, model_norm)."""
    w     = 1.0 / sig_d
    reg   = lam * (L if L is not None else np.eye(G.shape[1]))
    G_aug = np.vstack([G * w[:, None], reg])
    d_aug = np.concatenate([d_obs * w, np.zeros(reg.shape[0])])
    if nonneg:
        m, _ = nnls(G_aug, d_aug)
    else:
        m, _, _, _ = lstsq(G_aug, d_aug)
    d_pred = G @ m
    nrms   = np.sqrt(np.mean(((d_obs - d_pred) / sig_d)**2))
    mnorm  = float(np.linalg.norm(L @ m if L is not None else m))
    return m, d_pred, nrms, mnorm

def run_lcurve(G, d_obs, sig_d, lambdas, L=None):
    misfits, norms, sols = [], [], []
    for lam in lambdas:
        m, _, nrms, nm = solve_regularized(G, d_obs, sig_d, lam, L=L)
        misfits.append(nrms); norms.append(nm); sols.append(m)
    return np.array(misfits), np.array(norms), sols

def lcurve_corner(misfits, norms):
    """Triangle method (Hansen 1992): index of max distance from diagonal."""
    lm = np.log10(misfits); ln = np.log10(norms)
    lm = (lm - lm.min()) / (lm.max() - lm.min() + 1e-30)
    ln = (ln - ln.min()) / (ln.max() - ln.min() + 1e-30)
    dx = lm[-1] - lm[0]; dy = ln[-1] - ln[0]
    dist = np.abs(dy*(lm - lm[0]) - dx*(ln - ln[0])) / (np.hypot(dx, dy) + 1e-30)
    return int(np.argmax(dist))

# ── Mesh parser (Lab 9) ───────────────────────────────────────────────────────
def parse_gmsh41(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    def sec(tag): return next(i for i, l in enumerate(lines) if l.strip() == tag)
    ns      = sec("$Nodes")
    n_nodes = int(lines[ns+1].split()[1])
    coords  = np.array([list(map(float, lines[ns+3+n_nodes+i].split()))
                        for i in range(n_nodes)])
    es      = sec("$Elements")
    n_elems = int(lines[es+1].split()[1])
    tri_idx = np.array([[int(x)-1 for x in lines[es+3+i].split()[1:4]]
                        for i in range(n_elems)], dtype=np.int32)
    return coords, tri_idx

# ── Great-circle distance ─────────────────────────────────────────────────────
def distance_km(lat1, lon1, lat2, lon2):
    R  = 6371.0
    p1 = np.radians(lat1); p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1); dl = np.radians(lon2 - lon1)
    a  = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

PROJ = ccrs.PlateCarree()

def setup_map(ax, extent, gridlines=True):
    """Apply standard Cascadia map features to a cartopy axes."""
    ax.set_extent(extent, crs=PROJ)
    ax.add_feature(cfeature.LAND,      facecolor='0.93',   zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#cde8f0', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=0.7, edgecolor='0.25', zorder=2)
    ax.add_feature(cfeature.BORDERS,   lw=0.6, edgecolor='0.25',
                   linestyle='--', zorder=2)
    ax.add_feature(cfeature.STATES,    lw=0.4, edgecolor='0.45', zorder=2)
    if gridlines:
        gl = ax.gridlines(draw_labels=True, lw=0.3, color='gray', alpha=0.4)
        gl.top_labels = False; gl.right_labels = False
    return ax

print("Setup complete.")


### Download datasets

In [ ]:
# ── GNSS time series from GitHub ─────────────────────────────────────────────
GNSS_URL = (
    "https://raw.githubusercontent.com/amtseismo/EPS166/"
    "2d979cf169d1fcbf0542b765a147e472a4ca361d/"
    "datasets/gnss_SOPAC_trend_2010_2025.nc"
)
print("Downloading GNSS time series …")
resp = requests.get(GNSS_URL, timeout=180)
resp.raise_for_status()
ds = xr.open_dataset(io.BytesIO(resp.content), engine="h5netcdf")
print(f"  {ds.sizes['station']} stations, "
      f"{pd.to_datetime(ds.time.values).min().date()} – "
      f"{pd.to_datetime(ds.time.values).max().date()}")

# ── Cascadia mesh (same files as Lab 9) ───────────────────────────────────────
COMMIT_CSZ = "ef784d010afa83b3fef6a9b0a0c0a502ec981a2a"
BASE_CSZ   = (f"https://raw.githubusercontent.com/amtseismo/EPS166/"
              f"{COMMIT_CSZ}/datasets/CSZ/")

for fname in ["mccrory_latlon.msh"]:
    if not os.path.exists(fname):
        print(f"  Downloading {fname} …")
        urllib.request.urlretrieve(BASE_CSZ + fname, fname)
    else:
        print(f"  {fname} already present.")
print("All files ready.")


### Diagnostic — all stations in the dataset

Run this once to see every station in the GNSS dataset. Stations inside the 2024 ETS tremor corridor are highlighted. Use this map to adjust `TARGET_STATIONS` or the bounding box.

In [ ]:
fig = plt.figure(figsize=(11, 10))
ax = fig.add_subplot(1, 1, 1, projection=PROJ)
setup_map(ax, extent=[-132, -112, 40, 53])

# All stations
ax.scatter(all_lons, all_lats, s=12, color='0.75', zorder=3,
           transform=PROJ, label='All stations')

# ETS corridor stations
ax.scatter(all_lons[in_ets], all_lats[in_ets], s=30, color='steelblue',
           zorder=4, transform=PROJ, label=f'ETS corridor ({in_ets.sum()})')

for name, lon, lat in zip(
        [n for n, m in zip(all_names, in_ets) if m],
        all_lons[in_ets], all_lats[in_ets]):
    ax.text(lon + 0.1, lat, name, fontsize=5.5, va='center',
            color='navy', zorder=5, transform=PROJ)

# ETS bounding box
import matplotlib.patches as mpatches
rect = mpatches.Rectangle(
    (ETS_BOX['lon_min'], ETS_BOX['lat_min']),
    ETS_BOX['lon_max'] - ETS_BOX['lon_min'],
    ETS_BOX['lat_max'] - ETS_BOX['lat_min'],
    lw=1.5, edgecolor='steelblue', facecolor='none',
    linestyle='--', transform=PROJ, zorder=6)
ax.add_patch(rect)

ax.set_title(f'All GNSS stations in dataset  (n = {len(all_names)})\n'
             f'Blue = inside ETS corridor ({in_ets.sum()} stations)', fontsize=11)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

print(f"\nStations inside ETS corridor ({in_ets.sum()} total):")
cols = 8
for i in range(0, len(corridor_names), cols):
    print("  " + "  ".join(f"{n:8s}" for n in corridor_names[i:i+cols]))

---

## Part I: Extract SSE offsets

### Prediction

Before fitting any models, consider what you already know about Cascadia ETS:

1. Which GNSS component (east, north, or up) should show the clearest SSE signal at a coastal Cascadia station? In which direction should the displacement point, and is that the same as or opposite to the interseismic motion?
2. The 2024 ETS ran approximately 31 days. If you hadn't been told there was an event, how long a residual record would you need to see before confidently calling the transient "real" rather than seasonal noise or a monument problem?

Write your predictions before running the cells below.

### I.1 — Select Cascadia stations

The 2024 ETS tremor corridor runs from southern Vancouver Island through the Olympic Peninsula to southern Washington (PNSN tremor map, Malone 2025). Both coastal stations and **inland Puget Sound stations** are important: coastal stations are most sensitive to shallow and near-trench slip, while inland stations provide along-dip resolution and help distinguish the ETS depth range from shallower sources.


In [ ]:
# ── Target stations: ETS monitors along the 2024 tremor corridor ─────────────
# The 2024 event ran NW→SE from southern Vancouver Island through the Olympic
# Peninsula to southern Washington (PNSN tremor map, Malone 2025).
# Include both coastal AND inland Puget Sound stations — the ETS slip zone
# extends inland along the thrust dip direction.
TARGET_STATIONS = [
    # Southern Vancouver Island / BC
    "ALBH", "NANO", "PGC5", "PTAA", "HOLB", "BAMF", "NKLK",
    # Cape Flattery / northern Olympic Peninsula
    "NEAH", "P160", "P401", "P403",
    # Central Olympic Peninsula / coast
    "P435", "P162", "PABH", "P441",
    # Puget Sound corridor (inland — important for along-dip resolution)
    "SC02", "BLYN", "SEAT", "LKCP", "FISW", "P066",
    # Southern Washington
    "P160", "P162",
]
TARGET_STATIONS = list(dict.fromkeys(TARGET_STATIONS))  # deduplicate

# ── Case-insensitive match against the dataset ────────────────────────────────
# The dataset may store names as uppercase ("ALBH"), lowercase ("albh"),
# or mixed — normalise before comparing.
ds_names    = {str(s).upper(): str(s) for s in ds.station.values}
available   = [ds_names[s.upper()] for s in TARGET_STATIONS if s.upper() in ds_names]
found_target = len(available)

# Also add any stations in the 2024 tremor corridor bounding box
LAT_BOX = (45.5, 51.0); LON_BOX = (-130.0, -119.5)
in_box   = ((ds.lat.values >= LAT_BOX[0]) & (ds.lat.values <= LAT_BOX[1]) &
            (ds.lon.values >= LON_BOX[0]) & (ds.lon.values <= LON_BOX[1]))
box_add  = [str(s) for s in ds.station.values[in_box]
            if str(s) not in available]
available = available + box_add

print(f"Target stations matched in dataset: {found_target} / {len(TARGET_STATIONS)}")
print(f"Bounding-box additions:             {len(box_add)}")
print(f"Total candidate stations:           {len(available)}")
print()
# Diagnostic: show all matched stations with coordinates
print(f"{'Station':8s} {'Lon':>9s} {'Lat':>8s}")
print("-" * 30)
for s in available[:30]:   # cap at 30 to keep output readable
    st = ds.sel(station=s)
    print(f"{s:8s} {float(st.lon.values):9.3f} {float(st.lat.values):8.3f}")
if len(available) > 30:
    print(f"  … and {len(available)-30} more")


### I.2 — Fit trend + seasonal and extract SSE offsets

As in Lab 2: fit a linear + annual + semiannual model to data *before* the event window, project it forward, and measure the SSE displacement as

$$\delta = \overline{r}_{\text{post}} - \overline{r}_{\text{pre}}$$

Uncertainty propagation: if the daily scatter is $\sigma_{\text{day}}$ and we average over $n$ days,

$$\sigma_{\delta} = \sigma_{\text{day}}\sqrt{\dfrac{1}{n_{\text{post}}} + \dfrac{1}{n_{\text{pre}}}}$$

These $\sigma$ values fill the diagonal of the data covariance matrix $\mathbf{C}_d$ used to weight the inversion.

### Diagnostic — data coverage and raw offsets

Run this before the offset cell to understand what the dataset contains and why stations may be failing the SNR filter.

In [ ]:
import traceback

SSE_START   = pd.Timestamp("2024-08-06")
SSE_END     = pd.Timestamp("2024-09-06")
FIT_END     = pd.Timestamp("2024-07-01")
WINDOW_DAYS = 30

# ── 1. Dataset time range ─────────────────────────────────────────────────────
times_all = pd.to_datetime(ds.time.values)
print(f"Dataset time range: {times_all.min().date()} → {times_all.max().date()}")
print(f"SSE window:         {SSE_START.date()} → {SSE_END.date()}")
pre_start  = SSE_START - pd.Timedelta(days=WINDOW_DAYS)
post_end   = SSE_END   + pd.Timedelta(days=WINDOW_DAYS)
print(f"Pre window:         {pre_start.date()} → {SSE_START.date()}")
print(f"Post window:        {SSE_END.date()} → {post_end.date()}")
print()

# ── 2. Loop through ALL available stations and log what happens ───────────────
print(f"{'Station':8s} {'pre_fit':>8s} {'pre_win':>8s} {'post_win':>8s} "
      f"{'dE mm':>8s} {'dN mm':>8s} {'SNR':>6s} {'status'}")
print("-" * 72)

raw_offsets = {}
for sta in available:
    try:
        s     = ds.sel(station=sta)
        times = pd.to_datetime(s.time.values)
        t_yr  = s.dec_year.values
        e_mm  = 1000 * s.east_m.values
        n_mm  = 1000 * s.north_m.values
        se    = 1000 * s.east_sigma_m.values
        sn    = 1000 * s.north_sigma_m.values

        pre_fit  = times < FIT_END
        pre_m    = (times >= pre_start) & (times < SSE_START)
        post_m   = (times >= SSE_END)   & (times < post_end)

        n_prefit = int(np.sum(pre_fit & np.isfinite(e_mm)))
        n_pre    = int(np.sum(pre_m  & np.isfinite(e_mm)))
        n_post   = int(np.sum(post_m & np.isfinite(e_mm)))

        if n_prefit < 365:
            print(f"{sta:8s} {n_prefit:8d} {n_pre:8d} {n_post:8d} "
                  f"{'':>8s} {'':>8s} {'':>6s}  SKIP: <1yr pre-fit data")
            continue
        if n_pre < 7:
            print(f"{sta:8s} {n_prefit:8d} {n_pre:8d} {n_post:8d} "
                  f"{'':>8s} {'':>8s} {'':>6s}  SKIP: <7 days pre-window")
            continue
        if n_post < 7:
            print(f"{sta:8s} {n_prefit:8d} {n_pre:8d} {n_post:8d} "
                  f"{'':>8s} {'':>8s} {'':>6s}  SKIP: <7 days post-window")
            continue

        # Simple mean offset — no model fitting, just raw pre/post difference
        de_raw = np.nanmean(e_mm[post_m]) - np.nanmean(e_mm[pre_m])
        dn_raw = np.nanmean(n_mm[post_m]) - np.nanmean(n_mm[pre_m])
        sig_e  = max(np.nanstd(e_mm[pre_m | post_m]) *
                     np.sqrt(1/n_post + 1/n_pre), 0.3)
        sig_n  = max(np.nanstd(n_mm[pre_m | post_m]) *
                     np.sqrt(1/n_post + 1/n_pre), 0.3)
        snr    = max(abs(de_raw)/sig_e, abs(dn_raw)/sig_n)

        raw_offsets[sta] = dict(de=de_raw, dn=dn_raw, se=sig_e, sn=sig_n,
                                snr=snr, lon=float(s.lon), lat=float(s.lat))
        print(f"{sta:8s} {n_prefit:8d} {n_pre:8d} {n_post:8d} "
              f"{de_raw:8.2f} {dn_raw:8.2f} {snr:6.1f}  OK")

    except Exception:
        print(f"{sta:8s} {'':>8s} {'':>8s} {'':>8s} "
              f"{'':>8s} {'':>8s} {'':>6s}  ERROR: {traceback.format_exc().splitlines()[-1]}")

print()
print(f"Stations with any data: {len(raw_offsets)}")
print(f"Stations with SNR > 1:  {sum(1 for v in raw_offsets.values() if v['snr'] > 1.0)}")
print(f"Stations with SNR > 1.5:{sum(1 for v in raw_offsets.values() if v['snr'] > 1.5)}")


In [ ]:
SSE_START   = pd.Timestamp("2024-08-06")
SSE_END     = pd.Timestamp("2024-09-06")
WINDOW_DAYS = 30

offsets = {}

for sta in available:
    try:
        s     = ds.sel(station=sta)
        times = pd.to_datetime(s.time.values)
        t_yr  = s.dec_year.values
        e_mm  = 1000 * s.east_m.values
        n_mm  = 1000 * s.north_m.values
    except Exception:
        continue

    pre_m  = (times >= SSE_START - pd.Timedelta(days=WINDOW_DAYS)) & (times < SSE_START)
    post_m = (times >= SSE_END)   & (times < SSE_END + pd.Timedelta(days=WINDOW_DAYS))

    if pre_m.sum() < 7 or post_m.sum() < 7:
        continue

    # Fit secular + seasonal model on all data OUTSIDE the SSE window.
    # - Exclude a 5-day buffer on each side of the event.
    # - Use UNIT weights (not formal sigmas from the dataset, which can destabilise
    #   the fit). Sigma is instead estimated from the scatter in the pre/post windows.
    # - Fix t0 to the mean of the fitting epoch so the same reference epoch is used
    #   when evaluating on the full record (avoids a systematic constant offset).
    fit_m = (
        (times < SSE_START - pd.Timedelta(days=5)) |
        (times > SSE_END   + pd.Timedelta(days=5))
    )

    def secular_residual(vals, mask):
        """Unit-weighted secular+seasonal fit on mask; evaluate on full record."""
        valid = mask & np.isfinite(vals)
        if valid.sum() < 180:          # need at least 6 months
            return None
        t0  = float(t_yr[valid].mean())
        G_f = build_design_matrix(t_yr[valid], t0=t0)
        p, _, _, _ = lstsq(G_f, vals[valid])
        G_a = build_design_matrix(t_yr, t0=t0)
        return vals - G_a @ p

    res_e = secular_residual(e_mm, fit_m)
    res_n = secular_residual(n_mm, fit_m)
    if res_e is None or res_n is None:
        continue

    de = float(np.nanmean(res_e[post_m]) - np.nanmean(res_e[pre_m]))
    dn = float(np.nanmean(res_n[post_m]) - np.nanmean(res_n[pre_m]))

    # Sigma from scatter in the pre + post windows (robust, no dataset sigma needed)
    n_pre  = int(pre_m.sum());  n_post = int(post_m.sum())
    sig_e  = float(np.nanstd(res_e[pre_m | post_m]) * np.sqrt(1/n_post + 1/n_pre))
    sig_n  = float(np.nanstd(res_n[pre_m | post_m]) * np.sqrt(1/n_post + 1/n_pre))
    sig_e  = max(sig_e, 0.3); sig_n = max(sig_n, 0.3)

    offsets[sta] = dict(
        de=de, dn=dn, se=sig_e, sn=sig_n,
        snr=max(abs(de)/sig_e, abs(dn)/sig_n),
        lon=float(s.lon.values), lat=float(s.lat.values),
    )

# ── Filter by SNR and longitude ───────────────────────────────────────────────
MIN_SNR  = 1.0
LON_MAX  = -122.0   # exclude stations east of this — too far from the fault
offsets  = {k: v for k, v in offsets.items()
            if v["snr"] >= MIN_SNR and v["lon"] <= LON_MAX}
sta_list = list(offsets.keys())
print(f"Stations with SSE signal (SNR ≥ {MIN_SNR}): {len(sta_list)}")
print()
print(f"{'Station':8s} {'dE (mm)':>9s} {'dN (mm)':>9s} "
      f"{'σE (mm)':>8s} {'σN (mm)':>8s} {'SNR':>5s}")
print("-" * 50)
for sta in sta_list:
    v = offsets[sta]
    print(f"{sta:8s} {v['de']:9.2f} {v['dn']:9.2f} "
          f"{v['se']:8.3f} {v['sn']:8.3f} {v['snr']:5.1f}")

### I.3 — Displacement map

In [ ]:
ARROW_SCALE = 0.10   # degrees per mm — increase to make arrows more visible
mags = np.array([np.hypot(v['de'], v['dn']) for v in offsets.values()])

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(1, 1, 1, projection=PROJ)
setup_map(ax, extent=[-130, -119, 44.5, 52])

for sta, v in offsets.items():
    ax.quiver(v['lon'], v['lat'],
              v['de'] * ARROW_SCALE, v['dn'] * ARROW_SCALE,
              transform=PROJ, scale=1, scale_units='xy', angles='xy',
              color='steelblue', width=0.005, headwidth=5, zorder=4)
    ax.text(v['lon'] + 0.1, v['lat'], sta.upper(), fontsize=7,
            transform=PROJ, zorder=5)

# Reference arrow
ref_mm = 3.0
ax.quiver(-128.5, 45.8, ref_mm * ARROW_SCALE, 0,
          transform=PROJ, scale=1, scale_units='xy', angles='xy',
          color='k', width=0.005)
ax.text(-128.5, 45.6, f'{ref_mm:.0f} mm', fontsize=8, transform=PROJ)

ax.set_title(f'2024 Cascadia ETS — GNSS horizontal offsets\n'
             f'(trend + seasonal removed, {WINDOW_DAYS}-day pre/post windows)')
plt.tight_layout()
plt.show()

> **Read the residuals:**
> 1. What is the dominant direction of horizontal displacement? Is it in the same direction as or opposite to the interseismic motion at Cascadia coastal stations? Explain in terms of the elastic strain accumulated by fault locking.
> 2. Which station shows the largest displacement? Based on the arrow pattern, roughly where on the fault must the slip centre be?
> 3. How large are the per-station uncertainties relative to the signal? Which stations have the best signal-to-noise ratio, and where are they located?
> 4. We use only east and north in the inversion. What additional information would the up component provide, and why is it sometimes excluded despite being sensitive to thrust slip?


---

## Part II: Cascadia fault mesh

We use the same McCrory et al. (2012) triangulated Cascadia interface from Lab 9. We restrict the inversion to **46–51.5°N at 20–45 km depth** — the northern Cascadia ETS zone — and convert the mesh to local Cartesian metres as required by `cutde`.


In [ ]:
coords, tri_idx = parse_gmsh41("mccrory_latlon.msh")
verts3d   = coords[tri_idx]              # (N_tris, 3, [lon, lat, depth_km])
centroids = verts3d.mean(axis=1)
patch_lon = centroids[:, 0]
patch_lat = centroids[:, 1]
depth_km  = -centroids[:, 2]            # positive downward

print(f"Full mesh: {len(coords)} nodes, {len(tri_idx)} triangular patches")
print(f"  Lat range:   {patch_lat.min():.1f} – {patch_lat.max():.1f} °N")
print(f"  Depth range: {depth_km.min():.1f} – {depth_km.max():.1f} km")

# ── Filter to ETS zone ────────────────────────────────────────────────────────
# Tremor map (PNSN, Malone 2025) shows the 2024 event from southern
# Vancouver Island (~49.5°N) to southern Washington (~46.5°N) at ETS depths.
# Tighten the lat range to that corridor; keep the full depth range.
ETS_LAT_MIN, ETS_LAT_MAX   = 46.0, 50.5
ETS_DEP_MIN, ETS_DEP_MAX   = 20.0, 65.0

ets_mask     = ((patch_lat >= ETS_LAT_MIN) & (patch_lat <= ETS_LAT_MAX) &
                (depth_km  >= ETS_DEP_MIN) & (depth_km  <= ETS_DEP_MAX))
ets_tri_idx  = tri_idx[ets_mask]
N_PATCHES    = ets_mask.sum()
cent_lon_ets = patch_lon[ets_mask]
cent_lat_ets = patch_lat[ets_mask]
cent_dep_ets = depth_km[ets_mask]

print(f"\nETS zone ({ETS_LAT_MIN}–{ETS_LAT_MAX}°N, "
      f"{ETS_DEP_MIN}–{ETS_DEP_MAX} km): {N_PATCHES} patches")
print(f"Station count: {len(sta_list)}")
print(f"Observations M = {2*len(sta_list)}  (E + N at each station)")
print(f"Parameters   N = {N_PATCHES}")
print(f"Ratio M/N = {2*len(sta_list)/N_PATCHES:.2f} "
      f"({'over' if 2*len(sta_list) > N_PATCHES else 'under'}determined on paper; "
      f"effective rank is much lower)")

# ── Convert mesh to local Cartesian metres for cutde ─────────────────────────
node_x, node_y = lonlat_to_m(coords[:, 0], coords[:, 1])
node_z   = coords[:, 2] * 1000.0        # km → m (negative = below surface)
node_xyz = np.column_stack([node_x, node_y, node_z])
tris_m   = node_xyz[ets_tri_idx]        # (N_PATCHES, 3, 3) in metres

# ── Patch centroids in metres (for Laplacian) ─────────────────────────────────
cx_m, cy_m = lonlat_to_m(cent_lon_ets, cent_lat_ets)
cents_m    = np.column_stack([cx_m, cy_m, -cent_dep_ets*1000])

# ── Map ───────────────────────────────────────────────────────────────────────
# ── Map ───────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(1, 1, 1, projection=PROJ)
setup_map(ax, extent=[-130, -119, 44, 52])
sc = ax.scatter(cent_lon_ets, cent_lat_ets, c=cent_dep_ets,
                cmap='plasma_r', s=6, zorder=3, transform=PROJ)
plt.colorbar(sc, ax=ax, label='Depth (km)', shrink=0.7)
for sta in sta_list:
    v = offsets[sta]
    ax.plot(v['lon'], v['lat'], 'k^', ms=6, zorder=5, transform=PROJ)
    ax.text(v['lon']+0.1, v['lat'], sta.upper(), fontsize=7,
            transform=PROJ, zorder=5)
ax.set_title(f'McCrory Cascadia mesh — ETS inversion zone\n'
             f'{N_PATCHES} patches, {ETS_LAT_MIN}–{ETS_LAT_MAX}°N, '
             f'{ETS_DEP_MIN}–{ETS_DEP_MAX} km depth')
plt.tight_layout(); plt.show()

> **Examine the fault geometry:**
> 1. Is the system overdetermined or underdetermined on paper? Recall from Lab 10 why the effective rank of **G** can be far below both the row and column counts — particularly for deep fault patches.
> 2. Cascadia ETS is typically at 25–40 km depth. Why do we include shallower patches (20–25 km) in the inversion? What risk does this introduce?
> 3. The mesh coordinates are in lon/lat/km. Why do we convert to local Cartesian metres before calling `cutde`?


---

## Part III: Green's function matrix

### Prediction

Before building **G**, predict what it should look like:

1. Column $k$ of **G** is the surface displacement from 1 m of thrust slip on patch $k$. For a deep patch (35 km), do you expect a sharp or broad signal? How does this relate to why the inversion is underdetermined?
2. The stations are concentrated on the coast and inland Washington/BC. Which patches — shallow or deep, near or far from the coast — do you expect to have the highest-amplitude columns in **G**?

**G** has shape $(M \times N)$ where $M = 2N_{\text{stations}}$ (east and north interleaved) and $N = N_{\text{patches}}$. Column $k$ is the predicted east and north displacement (mm) from **1 m of pure thrust slip** on patch $k$.


In [ ]:
N_STA  = len(sta_list)
M_OBS  = 2 * N_STA

# ── Observation points in local Cartesian metres ──────────────────────────────
obs_x, obs_y = lonlat_to_m(
    [offsets[s]['lon'] for s in sta_list],
    [offsets[s]['lat'] for s in sta_list]
)
obs_xyz = np.column_stack([obs_x, obs_y, np.zeros(N_STA)])

# ── Data vector and uncertainty vector ───────────────────────────────────────
d_obs = np.zeros(M_OBS); sig_d = np.zeros(M_OBS)
for i, sta in enumerate(sta_list):
    v = offsets[sta]
    d_obs[2*i],   d_obs[2*i+1]   = v['de'], v['dn']
    sig_d[2*i],   sig_d[2*i+1]   = v['se'], v['sn']

print(f"Observations M = {M_OBS}  ({N_STA} stations × 2 components)")
print(f"Parameters   N = {N_PATCHES}")
print(f"σ range:  {sig_d.min():.3f} – {sig_d.max():.3f} mm")

# ── Build G (pure thrust slip = dip-slip +1) ──────────────────────────────────
# We tile the source triangle and slip vector to match the number of
# observation points, satisfying cutde's pairwise N-must-match requirement.
print(f"\nBuilding G ({M_OBS} × {N_PATCHES}) …")
G = np.zeros((M_OBS, N_PATCHES))

for k in range(N_PATCHES):
    tri_rep  = np.tile(tris_m[k],     (N_STA, 1, 1))   # (N_STA, 3, 3)
    slip_rep = np.tile([0., 1., 0.],  (N_STA, 1))       # (N_STA, 3) pure thrust
    disp     = hs.disp_free(obs_xyz, tri_rep, slip_rep, nu=NU)  # (N_STA, 3)
    G[0::2, k] = disp[:, 0] * 1000   # East  mm / m of slip
    G[1::2, k] = disp[:, 1] * 1000   # North mm / m of slip

print(f"G built. max|G| = {np.abs(G).max():.4f} mm/m")

# ── Sanity check: uniform 3 cm thrust slip ────────────────────────────────────
d_test = G @ (np.ones(N_PATCHES) * 0.03)
print(f"\nUniform 3 cm thrust — predicted E / N (mm):")
for i, sta in enumerate(sta_list):
    print(f"  {sta.upper():8s}  E = {d_test[2*i]:+.2f}   N = {d_test[2*i+1]:+.2f}")


> **Examine the system:**
> 1. What does each **row** of **G** represent physically? Each **column**?
> 2. For uniform thrust slip, the predicted east displacements should be negative (westward motion of the hanging wall above the ETS zone). Is this consistent with what you see — and with the SSE displacement direction you measured in Part I?
> 3. Why is the data covariance matrix diagonal here? Name a situation where you would need off-diagonal entries.
> 4. Building **G** required looping over all N_PATCHES patches. What would happen to the build time if you doubled the number of patches? How could you speed this up for a 10,000-patch global model?


---

## Part IV: Invert for slip

### Prediction

Before running the L-curve:

1. The real Cascadia dataset has spatially correlated noise (shared atmospheric delays, common-mode errors) that the synthetic Lab 10 data did not. Will this make the L-curve corner sharper or more diffuse? Why?
2. ETS patches at 20–45 km depth produce broad, overlapping surface signals. Based on what you saw in Lab 10, predict whether the optimal λ will be larger or smaller than in the Lab 10 synthetic case.

We solve:

$$\min_{\mathbf{m} \geq 0}\; \underbrace{\|\mathbf{W}(\mathbf{G}\mathbf{m} - \mathbf{d})\|^2}_{\text{weighted misfit}} + \underbrace{\lambda^2\|\mathbf{L}\mathbf{m}\|^2}_{\text{smoothing penalty}}$$

where $\mathbf{W} = \text{diag}(1/\sigma_i)$, $\mathbf{L}$ is a spatial graph Laplacian, and $\mathbf{m} \geq 0$ enforces thrust-only slip. The normalised RMS misfit $(\|\mathbf{W}(\mathbf{Gm}-\mathbf{d})\|/\sqrt{M})$ should be close to 1 at the optimal solution — fitting the data to within its noise level, no more.

### IV.1 — L-curve


In [ ]:
# ── Spatial Laplacian: neighbours within 35 km ───────────────────────────────
L_lap = build_spatial_laplacian(cents_m, radius_m=35e3)
print(f"Laplacian: mean neighbours per patch = "
      f"{L_lap.diagonal().mean():.1f} "
      f"(range {int(L_lap.diagonal().min())}–{int(L_lap.diagonal().max())})")

# ── L-curve sweep ─────────────────────────────────────────────────────────────
lambdas = np.logspace(-3, 1, 50)
print("Computing L-curve …")
misfits, norms, sols = run_lcurve(G, d_obs, sig_d, lambdas, L=L_lap)
corner  = lcurve_corner(misfits, norms)
LAM_OPT = lambdas[corner]
print(f"L-curve corner: λ = {LAM_OPT:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
sc = ax.scatter(misfits, norms, c=np.log10(lambdas), cmap='plasma', s=40)
plt.colorbar(sc, ax=ax, label='log₁₀(λ)')
ax.scatter(misfits[corner], norms[corner], s=250, c='k', marker='*',
           zorder=6, label=f'Corner  λ = {LAM_OPT:.4f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Normalised RMS misfit'); ax.set_ylabel('Roughness  ||Lm||')
ax.set_title('L-curve'); ax.legend(fontsize=9)

ax = axes[1]
ax.semilogx(lambdas, misfits, 'steelblue', lw=2, label='Normalised RMS misfit')
ax2 = ax.twinx()
ax2.semilogx(lambdas, norms, 'firebrick', lw=2, label='Roughness')
ax.axvline(LAM_OPT, color='k', ls='--', lw=1.5, label=f'λ = {LAM_OPT:.4f}')
ax.set_xlabel('λ'); ax.set_ylabel('Normalised RMS misfit', color='steelblue')
ax2.set_ylabel('Roughness  ||Lm||', color='firebrick')
ax.set_title('Misfit and roughness vs λ')
ax.legend(fontsize=8, loc='upper left')

plt.suptitle('L-curve — 2024 Cascadia ETS inversion', fontsize=11)
plt.tight_layout(); plt.show()

print("\nSet LAM_OPT above, or override it in the next cell.")


### IV.2 — Solve at optimal λ and plot the slip distribution

In [ ]:
# LAM_OPT was set by the L-curve corner; override here if desired.
# LAM_OPT = 0.1

m_opt, d_pred, nrms_opt, _ = solve_regularized(
    G, d_obs, sig_d, LAM_OPT, L=L_lap, nonneg=True)

print(f"Optimal λ:            {LAM_OPT:.4f}")
print(f"Normalised RMS misfit:{nrms_opt:.3f}  (target ≈ 1)")
print(f"Peak slip:            {m_opt.max()*100:.1f} cm")
print(f"Mean slip (> 5 mm):  "
      f"{m_opt[m_opt>0.005].mean()*100:.1f} cm" if (m_opt>0.005).any()
      else "  (no patch above 5 mm)")
print(f"Patches with slip > 1 cm: {(m_opt > 0.01).sum()}")

vmax = max(m_opt.max() * 100, 1.0)
fig = plt.figure(figsize=(14, 6))

# Left: geographic map of slip + observed and predicted arrows
ax1 = fig.add_subplot(1, 2, 1, projection=PROJ)
setup_map(ax1, extent=[-130, -119, 45, 52])
sc = ax1.scatter(cent_lon_ets, cent_lat_ets, c=m_opt * 100,
                 cmap='hot_r', s=10, vmin=0, vmax=vmax,
                 zorder=3, transform=PROJ)
plt.colorbar(sc, ax=ax1, label='Slip (cm)', shrink=0.7)

for i_s, sta in enumerate(sta_list):
    v = offsets[sta]
    ax1.quiver(v['lon'], v['lat'],
               v['de'] * ARROW_SCALE, v['dn'] * ARROW_SCALE,
               transform=PROJ, scale=1, scale_units='xy', angles='xy',
               color='steelblue', width=0.004, zorder=5,
               label='Observed' if i_s == 0 else '_')
    ax1.quiver(v['lon'], v['lat'],
               d_pred[2*i_s] * ARROW_SCALE, d_pred[2*i_s+1] * ARROW_SCALE,
               transform=PROJ, scale=1, scale_units='xy', angles='xy',
               color='firebrick', width=0.004, zorder=5,
               label='Predicted' if i_s == 0 else '_')

ax1.legend(fontsize=8, loc='lower left')
ax1.set_title(f'Slip distribution  (λ = {LAM_OPT:.4f})\nBlue = observed, Red = predicted')

# Right: depth vs latitude cross-section (plain axes)
ax2 = fig.add_subplot(1, 2, 2)
sc2 = ax2.scatter(cent_lat_ets, cent_dep_ets, c=m_opt * 100,
                  cmap='hot_r', s=15, vmin=0, vmax=vmax)
plt.colorbar(sc2, ax=ax2, label='Slip (cm)')
ax2.invert_yaxis()
ax2.set_xlabel('Latitude (°N)'); ax2.set_ylabel('Depth (km)')
ax2.set_title('Depth–latitude cross-section')
ax2.grid(True, alpha=0.25)

plt.suptitle('2024 Cascadia ETS — smoothed NNLS slip inversion', fontsize=11)
plt.tight_layout()
plt.show()

### IV.3 — Fit quality: observed vs predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, comp, idx in zip(axes, ['East', 'North'], [0, 1]):
    obs_c  = d_obs[idx::2]
    pred_c = d_pred[idx::2]
    lim    = max(np.abs(obs_c).max(), np.abs(pred_c).max()) * 1.2
    ax.errorbar(obs_c, pred_c, xerr=sig_d[idx::2],
                fmt='o', ms=6, color='steelblue', ecolor='0.65', capsize=3)
    for i, sta in enumerate(sta_list):
        ax.text(obs_c[i]+0.1, pred_c[i], sta.upper(), fontsize=7)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_xlabel(f'Observed {comp} (mm)'); ax.set_ylabel(f'Predicted {comp} (mm)')
    rms = np.sqrt(np.mean((obs_c - pred_c)**2))
    ax.set_title(f'{comp}  —  RMS = {rms:.2f} mm')
plt.suptitle('Observed vs predicted displacements', fontsize=11)
plt.tight_layout(); plt.show()


> **Evaluate the inversion:**
> 1. Where does the peak slip occur? Express the location in depth and latitude. Is this consistent with the typical depth range of northern Cascadia ETS (25–40 km)?
> 2. Is the normalised RMS misfit close to 1? If it is much less than 1, what does that imply about either the regularisation or the uncertainty estimates?
> 3. Which stations are most poorly fit in the observed-vs-predicted plot? Where are they located — could their larger residuals reflect a limitation of the station distribution or the fault geometry?
> 4. Try `LAM_OPT` at 10× and 1/10× the L-curve value. Describe how the slip distribution and normalised RMS misfit change in each case.


---

## Part V: Resolution matrix

The regularised inversion does not recover the true slip exactly — it recovers a blurred version. The **resolution matrix**

$$\mathbf{R} = (\mathbf{G}^T\mathbf{C}_d^{-1}\mathbf{G} + \lambda^2\mathbf{L}^T\mathbf{L})^{-1}\mathbf{G}^T\mathbf{C}_d^{-1}\mathbf{G}$$

tells you how much blurring has occurred. Row $j$ of $\mathbf{R}$ is the **resolution kernel** for patch $j$: the weights linking true slip across the fault to the estimated slip on that patch. When $\lambda = 0$, $\mathbf{R} = \mathbf{I}$ (perfect resolution, unstable inversion). As $\lambda \to \infty$, $\mathbf{R} \to \mathbf{0}$ (no information recovered).


In [ ]:
# Analytic R for the damped/smoothed problem (without NNLS — good approximation)
W_inv  = 1.0 / sig_d**2
GtWG   = G.T @ (G * W_inv[:, None])
reg_sq = LAM_OPT**2 * (L_lap.T @ L_lap)
A_inv  = np.linalg.inv(GtWG + reg_sq)
R      = A_inv @ GtWG
R_diag = np.diag(R)

print(f"Resolution matrix: {R.shape}")
print(f"Diagonal range:    {R_diag.min():.3f} – {R_diag.max():.3f}")
print(f"Well-resolved (R > 0.5):   {(R_diag > 0.5).sum()} patches")
print(f"Poorly resolved (R < 0.1): {(R_diag < 0.1).sum()} patches")

fig = plt.figure(figsize=(14, 6))

# Left: map of R diagonal
ax1 = fig.add_subplot(1, 2, 1, projection=PROJ)
setup_map(ax1, extent=[-130, -119, 44, 52])
sc = ax1.scatter(cent_lon_ets, cent_lat_ets, c=R_diag,
                 cmap='viridis', s=10, vmin=0, vmax=1,
                 zorder=3, transform=PROJ)
plt.colorbar(sc, ax=ax1, label='Self-resolution (diagonal of R)', shrink=0.7)
for sta in sta_list:
    ax1.plot(offsets[sta]['lon'], offsets[sta]['lat'],
             'k^', ms=5, zorder=5, transform=PROJ)
ax1.set_title(f'Resolution diagonal  (λ = {LAM_OPT:.4f})')

# Right: R diagonal vs depth, coloured by latitude (plain axes)
ax2 = fig.add_subplot(1, 2, 2)
sc2 = ax2.scatter(cent_dep_ets, R_diag, c=cent_lat_ets,
                  cmap='RdYlBu_r', s=15, alpha=0.7)
plt.colorbar(sc2, ax=ax2, label='Latitude (°N)')
ax2.axhline(0.5, color='k', ls='--', lw=1, label='R = 0.5')
ax2.set_xlabel('Depth (km)'); ax2.set_ylabel('Self-resolution')
ax2.set_title('Resolution vs depth'); ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

plt.suptitle('Resolution matrix — 2024 Cascadia ETS', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Resolution kernels for the best- and worst-resolved patches ──────────────
idx_best  = int(np.argmax(R_diag))
idx_worst = int(np.argmin(R_diag))

fig = plt.figure(figsize=(14, 6))
for col, idx, label in [(1, idx_best, 'Best resolved'),
                         (2, idx_worst, 'Worst resolved')]:
    ax = fig.add_subplot(1, 2, col, projection=PROJ)
    setup_map(ax, extent=[-130, -119, 44, 52])

    kernel = R[idx, :]
    vmax_k = max(np.abs(kernel).max(), 1e-6)
    sc = ax.scatter(cent_lon_ets, cent_lat_ets, c=kernel,
                    cmap='RdBu_r', s=10, vmin=-vmax_k, vmax=vmax_k,
                    zorder=3, transform=PROJ)
    plt.colorbar(sc, ax=ax, label='Kernel weight', shrink=0.7)
    ax.scatter(cent_lon_ets[idx], cent_lat_ets[idx],
               s=180, c='k', marker='*', zorder=6, transform=PROJ,
               label=f'Target  depth={cent_dep_ets[idx]:.0f} km  R={R_diag[idx]:.2f}')
    for sta in sta_list:
        ax.plot(offsets[sta]['lon'], offsets[sta]['lat'],
                'k^', ms=5, zorder=5, transform=PROJ)
    ax.set_title(f'{label} (★)  —  depth = {cent_dep_ets[idx]:.0f} km')
    ax.legend(fontsize=8, loc='lower left')

plt.suptitle('Resolution kernels — rows of R', fontsize=11)
plt.tight_layout()
plt.show()

> **Assess the resolution:**
> 1. Where are the best- and worst-resolved patches? Explain in terms of the station distribution and the depth dependence of patch fingerprints.
> 2. Do the high-slip patches from Part IV lie in well- or poorly-resolved regions? How does this affect your confidence in the result?
> 3. The best-resolved kernel should be compact (concentrated near the target patch). Is it? What would a perfect kernel look like?
> 4. A broad or negative-sidelobe kernel for the worst-resolved patch means the inversion is averaging over a large region. What are you actually reporting when you give a slip value for a poorly-resolved patch?
> 5. How does resolution vary with depth? Is this consistent with what you learned in Lab 10?


---

## Part VI: Moment, magnitude, and stress drop

### VI.1 — Seismic moment and moment magnitude

$$M_0 = \mu \sum_{i=1}^{N} A_i\, m_i \qquad \Longrightarrow \qquad M_w = \tfrac{2}{3}\log_{10}(M_0) - 6.07 \quad (M_0 \text{ in N·m})$$


In [ ]:
# ── Patch areas from triangle vertices (in metres) ────────────────────────────
def tri_area(verts):
    e1 = verts[1] - verts[0]; e2 = verts[2] - verts[0]
    return 0.5 * np.linalg.norm(np.cross(e1, e2))

patch_areas = np.array([tri_area(tris_m[k]) for k in range(N_PATCHES)])

# ── Seismic moment ────────────────────────────────────────────────────────────
M0 = MU * np.sum(patch_areas * m_opt)
Mw = (2/3) * np.log10(M0) - 6.07

slip_mask  = m_opt > 0.005   # patches with meaningful slip (> 5 mm)
A_slip     = patch_areas[slip_mask].sum()
mean_slip  = (patch_areas[slip_mask] * m_opt[slip_mask]).sum() / A_slip

print(f"Total ETS patch area:        {patch_areas.sum()/1e6:.0f} km²")
print(f"Slipping area (> 5 mm):      {A_slip/1e6:.0f} km²")
print(f"Area-weighted mean slip:      {mean_slip*100:.1f} cm")
print(f"Peak slip:                    {m_opt.max()*100:.1f} cm")
print(f"\nSeismic moment  M₀ = {M0:.3e} N·m")
print(f"Moment magnitude Mw = {Mw:.2f}")
print(f"\nPublished range for northern Cascadia ETS: Mw 5.5–6.8 (Bartlow 2020)")


### VI.2 — Stress drop

The **Brune (1970) circular-crack model** gives:

$$\Delta\sigma = \frac{7}{16}\frac{M_0}{r^3}, \qquad r = \sqrt{\frac{A_{\text{slip}}}{\pi}}$$

where $r$ is the radius of the equivalent circular crack and $A_{\text{slip}}$ is the total area of patches with meaningful slip. This gives a physical interpretation: stress drop $\approx \mu \times \langle\text{slip}\rangle / r$.


In [ ]:
r_eq          = np.sqrt(A_slip / np.pi)
delta_sigma   = (7/16) * M0 / r_eq**3    # Pa
delta_sig_kPa = delta_sigma / 1e3
delta_sig_MPa = delta_sigma / 1e6

print(f"Equivalent circular radius r = {r_eq/1e3:.1f} km")
print(f"Stress drop: {delta_sig_kPa:.1f} kPa  =  {delta_sig_MPa:.4f} MPa")

# ── Comparison table: ETS vs a regular earthquake of the same Mw ─────────────
# A typical crustal earthquake at the same Mw has a much smaller rupture area.
A_reg  = 300e6     # m² — compact seismogenic rupture for similar Mw
r_reg  = np.sqrt(A_reg / np.pi)
ds_reg = (7/16) * M0 / r_reg**3

print("\n" + "─"*62)
print(f"{'Property':<36} {'This ETS':>12} {'Regular Mw~same':>12}")
print("─"*62)
print(f"{'M₀ (N·m)':<36} {M0:>12.2e} {M0:>12.2e}")
print(f"{'Mw':<36} {Mw:>12.2f} {Mw:>12.2f}")
print(f"{'Rupture area (km²)':<36} {A_slip/1e6:>12.0f} {A_reg/1e6:>12.0f}")
print(f"{'Equiv. radius r (km)':<36} {r_eq/1e3:>12.1f} {r_reg/1e3:>12.1f}")
print(f"{'Stress drop (kPa)':<36} {delta_sig_kPa:>12.1f} {ds_reg/1e3:>12.1f}")
print(f"{'Stress drop (MPa)':<36} {delta_sig_MPa:>12.4f} {ds_reg/1e6:>12.2f}")
print(f"{'Stress drop ratio (regular / ETS)':<36} {ds_reg/delta_sigma:>12.0f}×")
print("─"*62)


### VI.3 — Interseismic slip deficit

In [ ]:
PLATE_RATE_MM_YR   = 40.0    # mm/yr — Juan de Fuca–N. America convergence
ETS_RECURRENCE_YR  = 14.0 / 12.0   # ~14-month recurrence interval

deficit_mm = PLATE_RATE_MM_YR * ETS_RECURRENCE_YR
fraction   = (mean_slip * 1000) / deficit_mm   # both in mm

print(f"Plate convergence rate:        {PLATE_RATE_MM_YR:.0f} mm/yr")
print(f"ETS recurrence interval:       {ETS_RECURRENCE_YR*12:.0f} months")
print(f"Interseismic deficit per cycle:{deficit_mm:.1f} mm  (fully-locked upper bound)")
print(f"ETS mean slip:                 {mean_slip*1000:.1f} mm")
print(f"ETS releases ≈ {fraction*100:.0f}% of the interseismic deficit at these depths")
print()
print("Note: ETS does NOT relieve stress on the locked shallow zone (0–20 km),")
print("which accumulates strain toward the next great earthquake.")


> **Interpret the source parameters:**
> 1. The ETS stress drop is orders of magnitude lower than a regular earthquake of the same moment magnitude. Physically, what does a low stress drop mean in terms of fault strength and driving stress? Why is this expected at 25–45 km depth in a subduction zone?
> 2. The Brune model assumes a circular crack with uniform stress drop. ETS slip distributions are elongated along-strike and narrow down-dip. Does using the circular model over- or under-estimate the stress drop? Sketch the geometry and explain.
> 3. Your Mw estimate should be compared against the published range Mw 5.5–6.8 for northern Cascadia ETS (Bartlow 2020). If it falls outside this range, what are three possible explanations?
> 4. ETS releases a significant fraction of the interseismic deficit at transition-zone depths. Does this mean Cascadia is "safe" because it regularly vents strain? What would you tell a policymaker?


---

## Synthesis

Write 3–5 sentences for each question.

> **1. The slip distribution**
> Describe where the slip concentrates and at what depth. Is this consistent with published northern Cascadia ETS locations? What are you most and least confident in, and why?

> **2. Regularisation and the L-curve**
> Explain what λ controls and why it cannot be zero. How did you choose your optimal λ, and how does that choice connect to the idea of fitting data to within its noise level? What happens at λ × 10 and λ / 10?

> **3. Data coverage and resolution**
> The station network is concentrated on Vancouver Island and the Washington coast. How does this limit resolution? Where would you add three new GNSS stations to most improve the inversion, and why?

> **4. Connecting to the seismic cycle**
> Cascadia ETS events recur every 12–14 months; each releases 1–5 cm of slip; the long-term convergence rate is ~40 mm/yr. What fraction of convergence is released as slow slip at transition-zone depths? How does your Mw estimate fit into the moment budget — and what does that imply for where the remaining strain accumulates?


---

## Summary

- **SSE offsets** are extracted by fitting a trend + seasonal model to pre-event data (as in Lab 2) and differencing 30-day pre/post-event windows. The per-station scatter propagates into diagonal entries of $\mathbf{C}_d$ that weight the inversion.

- The **McCrory et al. mesh** (same as Lab 9) provides a physically realistic triangulated fault geometry. ETS patches at 20–45 km depth over 46–51.5°N constitute the inversion domain. Converting from lon/lat/km to local Cartesian metres is required by `cutde`.

- **G** is built column by column using `cutde`: each column is the predicted east and north displacement (mm) from 1 m of pure thrust slip on one patch. The system is underdetermined in effective rank even when it appears overdetermined on paper.

- **Smoothed non-negative least squares** combines a weighted data-fit penalty, a spatial-Laplacian roughness penalty ($\lambda^2\|\mathbf{L}\mathbf{m}\|^2$), and non-negativity (thrust-only slip). The normalised RMS misfit at the L-curve optimal λ should be close to 1.

- The **resolution matrix** $\mathbf{R}$ shows that patches near the coastal station network are well resolved, while deep and offshore patches are poorly constrained — a fundamental limitation of land-based geodetic networks.

- The recovered slip gives $M_0 = \mu \sum A_i m_i$ and $M_w = \frac{2}{3}\log_{10}(M_0) - 6.07$. The **Brune stress drop** ($\Delta\sigma \sim 10$–$100$ kPa) is two to four orders of magnitude below typical crustal earthquakes — reflecting the near-ductile conditions at the deep subduction interface and distinguishing slow slip from regular earthquakes.

## References

- Bartlow, N. M. (2020). A long-term view of episodic tremor and slip in the Pacific Northwest. *GRL*, 47. https://doi.org/10.1029/2019GL085303
- Brune, J. N. (1970). Tectonic stress and spectra of seismic shear waves. *JGR*, 75(26), 4997–5009.
- Hanks, T. C., and Kanamori, H. (1979). A moment magnitude scale. *JGR*, 84(B5), 2348–2350.
- McCrory, P. A., et al. (2012). Juan de Fuca slab geometry. *JGR*, 117, B09306. https://doi.org/10.1029/2012JB009407
- Meade, B. J. (2007). Triangular dislocation elements. *Computers & Geosciences*, 33, 1064–1075.
- Rogers, G., and Dragert, H. (2003). Episodic tremor and slip on the Cascadia subduction zone. *Science*, 300(5627), 1942–1943.
